In [3]:
import pandas as pd


In [4]:
# Cargar datos desde el archivo CSV
from pathlib import Path
import pandas as pd

csv_path = Path("data/query_actividades_proc_q_actual.csv")
df = pd.read_csv(csv_path)

print(f"Archivo cargado desde: {csv_path}")
print(f"DataFrame cargado con {len(df)} filas y {len(df.columns)} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()
df.info()


Archivo cargado desde: data\query_actividades_proc_q_actual.csv
DataFrame cargado con 39033 filas y 5 columnas

Columnas: ['t1.id_epica', 't1.periodo', 'codigo_proceso', 't2.procedimiento', 't2.actividad']

Primeras filas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39033 entries, 0 to 39032
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   t1.id_epica       39033 non-null  float64
 1   t1.periodo        39033 non-null  object 
 2   codigo_proceso    39033 non-null  object 
 3   t2.procedimiento  39033 non-null  object 
 4   t2.actividad      39011 non-null  object 
dtypes: float64(1), object(4)
memory usage: 1.5+ MB


In [5]:
# Cargar arquetipos y hacer matching con los resultados SQL
import sys, os
# Asegura que la carpeta del notebook esté en sys.path
sys.path.insert(0, os.getcwd())

# Intentar import normal, con fallback a carga por ruta si falla
try:
    from helper.match_arquetipos import load_arquetipos, match_codes
except ModuleNotFoundError:
    import importlib.util
    mod_path = os.path.join(os.getcwd(), 'helper', 'match_arquetipos.py')
    spec = importlib.util.spec_from_file_location('match_arquetipos', mod_path)
    match_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(match_mod)
    load_arquetipos = match_mod.load_arquetipos
    match_codes = match_mod.match_codes

# Ajusta la ruta si es necesario
df_arq, df_arq_expl = load_arquetipos('data/arquetipos.csv')

# `df` debe existir en el notebook (resultado de la consulta SQL)
result, merged = match_codes(df, df_arq_expl, sql_code_col='codigo_proceso')
print(f"Filas SQL: {len(df)}; coincidencias en merged (filas con proc_code): {merged['proc_code'].notna().sum()}")
# Filtrar solo filas con coincidencias en arquetipos
result_filtered = result[result['matched_codigo_arquetipo'].notna()]
result_filtered

Filas SQL: 39033; coincidencias en merged (filas con proc_code): 29295


,t1.id_epica,t1.periodo,codigo_proceso,t2.procedimiento,t2.actividad,matched_proc_codes,matched_codigo_arquetipo,matched_nombre_arquetipo
122,6811752.0,2026Q1,PAN040102,Abrir Producto de Crédito,autorizar operaciones por callback,[PAN040102],[ARQ002],[Abrir producto]
123,6811752.0,2026Q1,PAN040102,Realizar cálculos Revolvente,recibir información [act266],[PAN040102],[ARQ002],[Abrir producto]
124,6811752.0,2026Q1,PAN040102,Realizar cálculos Revolvente,notificar inconsistencia de fecha,[PAN040102],[ARQ002],[Abrir producto]
125,6811752.0,2026Q1,PAN040102,Realizar cálculos Revolvente,m: cumplimiento de los servicios p: abrir prod...,[PAN040102],[ARQ002],[Abrir producto]
126,6811752.0,2026Q1,PAN040102,Realizar cálculos Revolvente,ingresar comprobante en sap,[PAN040102],[ARQ002],[Abrir producto]
...,...,...,...,...,...,...,...,...
39028,7056043.0,2026Q1,VAL080512,"Asesorar, negociar o promover operaciones con ...",c007: verificar que la orden cumpla la informa...,[T160265],[ARQ056],[Administrar el servicio de los canales]
39029,7056043.0,2026Q1,VAL080512,"Asesorar, negociar o promover operaciones con ...",c004: validar listas de control,[T160265],[ARQ056],[Administrar el servicio de los canales]
39030,7056043.0,2026Q1,VAL080512,"Asesorar, negociar o promover operaciones con ...",c003: validar condiciones del producto,[T160265],[ARQ056],[Administrar el servicio de los canales]
39031,7056043.0,2026Q1,VAL080512,"Asesorar, negociar o promover operaciones con ...",c001: verificar que el cliente se encuentre vi...,[T160265],[ARQ056],[Administrar el servicio de los canales]


In [6]:
# Guardar resultados en CSV
import os
from datetime import datetime

# Check if result_filtered exists and has data
if 'result_filtered' not in locals() or result_filtered is None or len(result_filtered) == 0:
	print("Advertencia: No hay resultados con coincidencias para guardar.")
else:
	output_dir = 'data/output'
	os.makedirs(output_dir, exist_ok=True)
	# Usar timestamp para evitar conflictos
	timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
	output_path = os.path.join(output_dir, f'resultado_matching_arquetipos_{timestamp}.csv')
	result_filtered.to_csv(output_path, index=False, encoding='utf-8')
	print(f"Archivo guardado en: {output_path}")
	print(f"Total de filas: {len(result_filtered)}")

Archivo guardado en: data/output\resultado_matching_arquetipos_20260618_140956.csv
Total de filas: 26548
